# Model Validation and Testing

**Goal:** Validate the finalized `CNN3Layer` implementation from `src/models.py`

In [3]:
# Colab bootstrap cell
!git clone https://github.com/kar137/ouroboros.git
%cd ouroboros

Cloning into 'ouroboros'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 15 (delta 3), reused 10 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 8.86 KiB | 8.86 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/ouroboros/ouroboros


In [4]:
import sys
import torch
import torch.nn as nn



from src.models import CNN3Layer, count_parameters, model_summary

print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"\n✓ Successfully imported CNN3Layer from src.models")

PyTorch Version: 2.9.0+cu126
Device: cuda

✓ Successfully imported CNN3Layer from src.models


## Test 1: CIFAR-10 Configuration

Validate RGB input (3×32×32) with 10 output classes.

In [5]:
print("=" * 70)
print("Test 1: CIFAR-10 (3 channels, 32×32, 10 classes)")
print("=" * 70)

# Instantiate model
model_cifar10 = CNN3Layer(num_classes=10, in_channels=3)
model_cifar10.eval()

# Create test batch
batch_size = 8
test_input = torch.randn(batch_size, 3, 32, 32)

# Forward pass
with torch.no_grad():
    output = model_cifar10(test_input)

# Validate
params = count_parameters(model_cifar10)
expected_output_shape = (batch_size, 10)
shape_match = output.shape == expected_output_shape
params_in_range = 47500 <= params <= 52500

print(f"\nInput Shape:           {tuple(test_input.shape)}")
print(f"Output Shape:          {tuple(output.shape)}")
print(f"Expected Shape:        {expected_output_shape}")
print(f"Shape Match:           {'✓ PASS' if shape_match else '✗ FAIL'}")
print(f"\nParameter Count:       {params:,}")
print(f"Target Range:          47,500 - 52,500")
print(f"Within Range:          {'✓ PASS' if params_in_range else '✗ FAIL'}")
print(f"\nOverall Status:        {'✓ PASS' if shape_match and params_in_range else '✗ FAIL'}")

Test 1: CIFAR-10 (3 channels, 32×32, 10 classes)

Input Shape:           (8, 3, 32, 32)
Output Shape:          (8, 10)
Expected Shape:        (8, 10)
Shape Match:           ✓ PASS

Parameter Count:       50,402
Target Range:          47,500 - 52,500
Within Range:          ✓ PASS

Overall Status:        ✓ PASS


## Test 2: CIFAR-100 Configuration

Validate RGB input (3×32×32) with 100 output classes.

In [6]:
print("=" * 70)
print("Test 2: CIFAR-100 (3 channels, 32×32, 100 classes)")
print("=" * 70)

# Instantiate model
model_cifar100 = CNN3Layer(num_classes=100, in_channels=3)
model_cifar100.eval()

# Create test batch
batch_size = 8
test_input = torch.randn(batch_size, 3, 32, 32)

# Forward pass
with torch.no_grad():
    output = model_cifar100(test_input)

# Validate
params = count_parameters(model_cifar100)
expected_output_shape = (batch_size, 100)
shape_match = output.shape == expected_output_shape

print(f"\nInput Shape:           {tuple(test_input.shape)}")
print(f"Output Shape:          {tuple(output.shape)}")
print(f"Expected Shape:        {expected_output_shape}")
print(f"Shape Match:           {'✓ PASS' if shape_match else '✗ FAIL'}")
print(f"\nParameter Count:       {params:,}")
print(f"Note:                  Slightly higher due to larger final layer (88→100 vs 88→10)")
print(f"\nOverall Status:        {'✓ PASS' if shape_match else '✗ FAIL'}")

Test 2: CIFAR-100 (3 channels, 32×32, 100 classes)

Input Shape:           (8, 3, 32, 32)
Output Shape:          (8, 100)
Expected Shape:        (8, 100)
Shape Match:           ✓ PASS

Parameter Count:       58,412
Note:                  Slightly higher due to larger final layer (88→100 vs 88→10)

Overall Status:        ✓ PASS


## Test 3: Fashion-MNIST Configuration

Validate grayscale input (1×28×28) with 10 output classes.

In [7]:
print("=" * 70)
print("Test 3: Fashion-MNIST (1 channel, 28×28, 10 classes)")
print("=" * 70)

# Instantiate model
model_fmnist = CNN3Layer(num_classes=10, in_channels=1)
model_fmnist.eval()

# create test batch
batch_size = 8
test_input = torch.randn(batch_size, 1, 28, 28)

# forward pass
with torch.no_grad():
    output = model_fmnist(test_input)

# validate
params = count_parameters(model_fmnist)
expected_output_shape = (batch_size, 10)
shape_match = output.shape == expected_output_shape
params_in_range = 47500 <= params <= 52500

print(f"\nInput Shape:           {tuple(test_input.shape)}")
print(f"Output Shape:          {tuple(output.shape)}")
print(f"Expected Shape:        {expected_output_shape}")
print(f"Shape Match:           {'✓ PASS' if shape_match else '✗ FAIL'}")
print(f"\nParameter Count:       {params:,}")
print(f"Target Range:          47,500 - 52,500")
print(f"Within Range:          {'✓ PASS' if params_in_range else '✗ FAIL'}")
print(f"Note:                  Slightly fewer params due to 1-channel input (vs 3-channel)")
print(f"\nOverall Status:        {'✓ PASS' if shape_match and params_in_range else '✗ FAIL'}")

Test 3: Fashion-MNIST (1 channel, 28×28, 10 classes)

Input Shape:           (8, 1, 28, 28)
Output Shape:          (8, 10)
Expected Shape:        (8, 10)
Shape Match:           ✓ PASS

Parameter Count:       49,970
Target Range:          47,500 - 52,500
Within Range:          ✓ PASS
Note:                  Slightly fewer params due to 1-channel input (vs 3-channel)

Overall Status:        ✓ PASS


## Test 4: Resolution Flexibility

Verify the model handles different input resolutions correctly (AdaptiveAvgPool2d).

In [8]:
print("=" * 70)
print("Test 4: Resolution Flexibility")
print("=" * 70)

model = CNN3Layer(num_classes=10, in_channels=3)
model.eval()

test_resolutions = [
    (3, 28, 28),
    (3, 32, 32),
    (3, 64, 64),
    (3, 224, 224),  # ImageNet size
]

print("\nTesting various input resolutions:")
print("-" * 70)

all_passed = True
for i, (c, h, w) in enumerate(test_resolutions, 1):
    test_input = torch.randn(1, c, h, w)
    
    with torch.no_grad():
        try:
            output = model(test_input)
            passed = output.shape == (1, 10)
            all_passed = all_passed and passed
            status = '✓ PASS' if passed else '✗ FAIL'
        except Exception as e:
            output = None
            passed = False
            all_passed = False
            status = f'✗ ERROR: {str(e)[:30]}'
    
    input_shape = tuple(test_input.shape)
    output_shape = tuple(output.shape) if output is not None else 'N/A'
    print(f"Test {i}: {input_shape} → {output_shape}  {status}")

print("-" * 70)
print(f"Overall Resolution Test: {'✓ PASS' if all_passed else '✗ FAIL'}")

Test 4: Resolution Flexibility

Testing various input resolutions:
----------------------------------------------------------------------
Test 1: (1, 3, 28, 28) → (1, 10)  ✓ PASS
Test 2: (1, 3, 32, 32) → (1, 10)  ✓ PASS
Test 3: (1, 3, 64, 64) → (1, 10)  ✓ PASS
Test 4: (1, 3, 224, 224) → (1, 10)  ✓ PASS
----------------------------------------------------------------------
Overall Resolution Test: ✓ PASS


## Detailed Model Summary

Using built-in `model_summary()` function to inspect layer-wise details.

In [9]:
# CIFAR-10 summary
model_cifar = CNN3Layer(num_classes=10, in_channels=3)
model_summary(model_cifar, (1, 3, 32, 32))


Model Summary - Input Shape: (1, 3, 32, 32)

Layer Name                     Output Shape                  
------------------------------------------------------------
conv1                          (1, 24, 32, 32)               
bn1                            (1, 24, 32, 32)               
relu1                          (1, 24, 32, 32)               
pool1                          (1, 24, 16, 16)               
conv2                          (1, 48, 16, 16)               
bn2                            (1, 48, 16, 16)               
relu2                          (1, 48, 16, 16)               
pool2                          (1, 48, 8, 8)                 
conv3                          (1, 88, 8, 8)                 
bn3                            (1, 88, 8, 8)                 
relu3                          (1, 88, 8, 8)                 
adaptive_pool                  (1, 88, 1, 1)                 
fc                             (1, 10)                       

Output Shape: (1, 10)
To

In [10]:
# Fashion-MNIST summary
model_fmnist = CNN3Layer(num_classes=10, in_channels=1)
model_summary(model_fmnist, (1, 1, 28, 28))


Model Summary - Input Shape: (1, 1, 28, 28)

Layer Name                     Output Shape                  
------------------------------------------------------------
conv1                          (1, 24, 28, 28)               
bn1                            (1, 24, 28, 28)               
relu1                          (1, 24, 28, 28)               
pool1                          (1, 24, 14, 14)               
conv2                          (1, 48, 14, 14)               
bn2                            (1, 48, 14, 14)               
relu2                          (1, 48, 14, 14)               
pool2                          (1, 48, 7, 7)                 
conv3                          (1, 88, 7, 7)                 
bn3                            (1, 88, 7, 7)                 
relu3                          (1, 88, 7, 7)                 
adaptive_pool                  (1, 88, 1, 1)                 
fc                             (1, 10)                       

Output Shape: (1, 10)
To

## Parameter Distribution Analysis

Visualize where parameters are concentrated in the architecture.

In [11]:
model = CNN3Layer(num_classes=10, in_channels=3)

print("\n" + "=" * 70)
print("Parameter Distribution by Layer (CIFAR-10 configuration)")
print("=" * 70)

# Group by layer type
layer_groups = {
    'Conv Layers': [],
    'BatchNorm Layers': [],
    'Classifier': []
}

for name, param in model.named_parameters():
    if param.requires_grad:
        num_params = param.numel()
        if 'conv' in name:
            layer_groups['Conv Layers'].append((name, param.shape, num_params))
        elif 'bn' in name:
            layer_groups['BatchNorm Layers'].append((name, param.shape, num_params))
        elif 'fc' in name:
            layer_groups['Classifier'].append((name, param.shape, num_params))

# Print grouped
grand_total = 0
for group_name, params_list in layer_groups.items():
    group_total = sum(p[2] for p in params_list)
    grand_total += group_total
    
    print(f"\n{group_name}:")
    print("-" * 70)
    for name, shape, count in params_list:
        print(f"  {name:<30} {str(shape):<20} {count:>8,}")
    print(f"  {'Subtotal:':<30} {'':<20} {group_total:>8,}")

print("\n" + "=" * 70)
print(f"{'TOTAL PARAMETERS:':<30} {'':<20} {grand_total:>8,}")
print("=" * 70)

# Calculate percentages
print("\nParameter Distribution by Group:")
print("-" * 70)
for group_name, params_list in layer_groups.items():
    group_total = sum(p[2] for p in params_list)
    percentage = (group_total / grand_total) * 100
    print(f"{group_name:<30} {group_total:>8,} ({percentage:>5.1f}%)")


Parameter Distribution by Layer (CIFAR-10 configuration)

Conv Layers:
----------------------------------------------------------------------
  conv1.weight                   torch.Size([24, 3, 3, 3])      648
  conv1.bias                     torch.Size([24])           24
  conv2.weight                   torch.Size([48, 24, 3, 3])   10,368
  conv2.bias                     torch.Size([48])           48
  conv3.weight                   torch.Size([88, 48, 3, 3])   38,016
  conv3.bias                     torch.Size([88])           88
  Subtotal:                                             49,192

BatchNorm Layers:
----------------------------------------------------------------------
  bn1.weight                     torch.Size([24])           24
  bn1.bias                       torch.Size([24])           24
  bn2.weight                     torch.Size([48])           48
  bn2.bias                       torch.Size([48])           48
  bn3.weight                     torch.Size([88])        